# Python Machine Learning Labs: Book Rating Predictions

### Introduction

In this project, ***.

### Data Import

In [ ]:
"""
Books ML Project
Analysis and machine learning on a dataset of books.
"""
import pandas as pd

In [ ]:
books = pd.read_csv('books.csv', on_bad_lines='warn')
print(books.columns.tolist())
print(books.shape)

In [ ]:
books.columns = books.columns.str.strip() # remove extra spaces in column names, notably num_pages

Upon importing the data with `pd.read_csv('books.csv', on_bad_lines='warn')`, the data frame is created but we know 4 rows are skipped for having an extra column. Python says these are rows 3350, 4704, 5879, and 8981. Let's see what those lines look like. 

In [ ]:
with open('books.csv', 'r', encoding='utf-8') as f:
    comma_lines = f.readlines()

# check the problematic lines (subtract 1 for 0-indexing)
for i in [3349, 4703, 5878, 8980]:
    print(f"Line {i+1}: {comma_lines[i]}")

We can see from this code the issue is due to an extra comma in the author column. It's much easier to see this issue when viewing the CSV in Excel, filtering this erroneous 13th column by all non-blank values to see these 4 rows. We can fix these rows manually and add them to our dataframe if we don't want to lose the data. 

In [ ]:
bad_lines = [3349, 4703, 5878, 8980]
manual_rows = []
with open('books.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()
    for i in bad_lines:
        line = lines[i].strip()
        parts = line.split(',', 12)
        if len(parts) == 13:
            parts = [parts[0], parts[1], parts[2] + parts[3], parts[4], parts[5], parts[6], parts[7], parts[8], parts[9], parts[10], parts[11], parts[12]]
        manual_rows.append(parts)

manual_df = pd.DataFrame(manual_rows, columns=books.columns)
manual_df # ensure it has the correct data

In [ ]:
# We confirmed the data is correct, so we can concatonate manual_df to books.
books = pd.concat([books, manual_df], ignore_index=True)
print(books.shape) # Print shape to confirm it is the expected 11127 rows, 12 columns.

### Data Exploration & Cleaning

We begin exploring the data to get a better idea of how it's structured. 

In [ ]:
books.head() # General view of the data.

In [ ]:
books.info()
books.describe()

If we look to our output from `books.describe()`, we can learn several things. For example, there is a difference in number of titles and uniqueness of titles, indicating there may be some duplicates throughout. Also, we see the datatypes are objects or strings, we may want to convert these to their proper types, like dates to datetime. The machine learning model will not use some features for the predictions, such as the bookID, isbn, and isbn13. Therefore, these columns can be removed. 

In [ ]:
books_less_columns = books.copy()
books_less_columns = books_less_columns.drop(columns=['bookID', 'isbn', 'isbn13'])

In [ ]:
books["publication_date"].isna().sum()

In [ ]:
books_dates = books_less_columns.copy()
books_dates["publication_date"] = pd.to_datetime(
    books_dates["publication_date"],
    errors="coerce"
    )
books_dates["publication_date"].isna().sum() # Check how many publication dates are invalid.

The change of the publication date column to datetime caused two rows to be invalid, let's view those original rows to see why.

In [ ]:
books[
    pd.to_datetime(
        books["publication_date"],
        errors="coerce"
    ).isna()
][["title", "publication_date"]]

It's because these aren't real dates, there are not 31 days in November or June. We could seek this data and manually fix it, but for consistency and reproducibility of the project they can be left as missing, since 2 rows of 11,000 won't make a meaningful difference. We will remove those 2 rows now. Additionally, we will add a column for just the year, since the month and day are less relevant to the model. As an additional step, we will calculate the age of the book to make the number more intuitive for the model instead of a larger number for year, the smaller "book_age" will have the same effect on the model. 

In [ ]:
books_dates = books_dates.dropna(subset=["publication_date"])
books_dates["publication_year"] = books_dates["publication_date"].dt.year
books_dates["book_age"] = 2026 - books_dates["publication_year"]
books_dates[["publication_year", "book_age"]] = books_dates[["publication_year", "book_age"]].astype(int)

In [ ]:
books_dates[["publication_date", "publication_year", "book_age"]].info() # Confirm the new columns are now the correct types and that there are no null values in those columns.

Now that we have fixed the dates, we can move along with the cleaning. Next, let's continue converting the objects to numeric types where appropriate. 

In [ ]:
books_numeric = books_dates.copy()
books_numeric.dtypes

We want to convert these columns `"average_rating", "num_pages", "ratings_count", "text_reviews_count"` into numeric type, but first we want to ensure there is not missing data to start and that at the end we get the same data out, just as `int64` or `float64` instead of `object`.

In [ ]:
cols = ["average_rating", "num_pages", "ratings_count", "text_reviews_count"]

for c in cols:
    coerced = pd.to_numeric(books_numeric[c], errors="coerce")
    
    print("\n", c)
    print("original nulls:", books_numeric[c].isna().sum())
    print("new nulls after coercion:", coerced.isna().sum())

We have no nulls to begin with and none after the conversion, so it's likely safe to convert. 

In [ ]:
books_numeric[cols] = books_numeric[cols].apply(pd.to_numeric)

In [ ]:
books_numeric[cols].dtypes # Checking the dtypes of the numeric columns to ensure they are now numeric types.

In [ ]:
books_numeric[cols].describe() # Checking the descriptive statistics of the numeric columns to ensure they are now numeric types and to get a sense of the data distribution.

With all of our data types corrected, we can now move our attention to the language code. 

In [ ]:
books_lang = books_numeric.copy()
books_lang["language_code"].unique()

In [ ]:
# First we can standardize the books in English to all be "eng".
books_lang["language_code"] = (
    books_lang["language_code"]
    .str.lower()
    .replace({
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)
books_lang["language_code"].value_counts()

There are many languages that have very few appearances. Therefore, we can keep the bigger categories, and put the smaller ones into a grouped "other" category. What constitutes bigger? For now, we will keep Japanese as the smallest category with 46, but even that is very few compared to the others. 

In [ ]:
top_langs = ["eng", "spa", "fre", "ger", "jpn"]

books_lang["language_code"] = books_lang["language_code"].apply(
    lambda x: x if x in top_langs else "other"
)

books_lang["language_code"].value_counts()